# **Respuesta en tiempo real**

>**Importante**   
Tenemos que tener completado el paso anterior "02-obtener-api-key.md"   
Además usaremos los métodos de la sección "04-peticion-multiturno".

In [ ]:
#Instalar dependencias
%pip install anthropic python-dotenv

In [1]:
#Guardar y cargar API KEY
#Para guardar la key ponemos un .env y dentro ANTHROPIC_API_KEY="NUESTRA KEY"
#Importante recordar poner el .env en el gitignore
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
#Crear un cliente API
from anthropic import Anthropic

cliente = Anthropic()
model = "claude-sonnet-5-5"

---

## Respuesta en tiempo real

La respuesta en tiempo real es importante para que el usuario pueda ver algo mientras carga la respuesta del modelo. Algunas respuestas pueden llegar a tardar hasta 10 segundos a 1 minutos.   
*Especial atencion al nuevo valor **stream** dentro de la función chat*

In [6]:

# 1. En esta función solicitamos la lista de mensajes totales y un texto para introducir un nuevo mensaje
# definimos el rol y contexto de forma rápida y añadimos con un append el mensaje nuevo a la lista total.
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

#2. Hacemos lo mismo con el mensaje de respuesta, como rol tendrá assistant
def add_assistant_message(messages, text):
    assistante_message = {"role": "assistant", "content": text}
    messages.append(assistante_message)

#3. Funcion automatizada en la que solicitamos la lista de mensajes y retornamos la respuesta habiéndo pasado el historial completo
def chat(messages, system):

    message = cliente.messages.create(
        system=system,
        model=model,
        max_tokens=1000,
        messages=messages 
    )
    for block in message.content:
        if block.type == "text":
            return block.text

In [ ]:
#Lista vacía de mensajes
messages = []

system="""
Eres un asistente de la aplicación Rutea, aplicación de rutas.
Antes de dar la respuesta da la bienvenida como asistente de Rutea.
Máximo 6 lineas.
"""

add_user_message(messages, "Dime la dificultad de la ruta Torla-Cola de caballo")

#Con .stream vemos que el usuario ve los chucks de texto de forma progresiva mientras este está siendo generado.
with cliente.messages.stream(
    system=system,
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")


¡Bienvenido a Rutea, tu asistente de rutas! 🥾

**Torla – Cola de Caballo: dificultad media (baja-media en el tramo principal).**
- Distancia: unos 17 km ida y vuelta desde la Pradera de Ordesa (más si sales andando desde Torla).
- Desnivel: unos 600-800 m, suave y con buen camino. Duración: 6-7 h.
- Es apta para personas con forma física normal. Lo más exigente es la distancia.
- En temporada alta, el acceso a la Pradera es en bus desde Torla. Consulta horarios y el tiempo.